##### sea ice volume change decomposition

In [7]:
"""
render_equations.py
===================
Renders all equations from sections 4, 5, and 6 of the SIV budget
methodology document as consistently-sized PNG images suitable for
inserting into a Word document.

Each equation is saved as eq_NN_<label>.png in ./outputs/equations/.
A reference sheet (eq_reference_sheet.png) is also produced showing
all equations with their numbers and labels for easy lookup.

Usage:
    python render_equations.py

Requirements:
    matplotlib (already installed in ROVER environment)

Output size / Word insertion:
    Each PNG is rendered at 200 DPI. Insert into Word at a fixed width
    of 12 cm for display equations; 8 cm for simpler single-line ones.
    All images have identical canvas height so they align consistently.
"""

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path

OUTPUT_DIR = Path('./outputs/equations')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Rendering parameters ─────────────────────────────────────────────────────
DPI         = 200        # resolution — increase to 300 for print-quality
FIG_WIDTH   = 7.0        # inches — all equations same width
FIG_HEIGHT  = 0.85       # inches — uniform height for consistency
FONTSIZE    = 16         # mathtext font size
BG_COLOR    = 'white'
TEXT_COLOR  = 'black'
PADDING     = 0.08       # fractional vertical padding (axes fraction)

# ── Equation definitions ─────────────────────────────────────────────────────
# Each entry: (number, label, latex_string)
# LaTeX uses matplotlib mathtext — standard LaTeX math syntax.

EQUATIONS = [

    # ── Section 4: Per-Cell SIV Budget Equation ──────────────────────────────
    (
        '4.1',
        'budget_equation',
        r'$\Delta \mathrm{SIV}_{\mathrm{cell}} '
        r'= F_{\mathrm{dynamic,\,cell}} + F_{\mathrm{thermo,\,cell}}$'
    ),

    # ── Section 5.1: Sea Ice Volume per Cell ─────────────────────────────────
    (
        '5.1',
        'siv_per_cell',
        r'$\mathrm{SIV}_{\mathrm{cell}}(t) '
        r'= \mathrm{siconc}(t) \times \mathrm{sithick}(t) \times A_{\mathrm{cell}}'
        r'$'
    ),
    (
        '5.2',
        'dsiv_per_cell',
        r'$\Delta \mathrm{SIV}_{\mathrm{cell}}(t) '
        r'= \mathrm{SIV}_{\mathrm{cell}}(t) - \mathrm{SIV}_{\mathrm{cell}}(t-1)'
        r'$'
    ),

    # ── Section 5.2: Ice Volume Flux Density ─────────────────────────────────
    (
        '5.3',
        'flux_density_x',
        r'$F_x = u_{\mathrm{si}} \times \mathrm{sithick} \times \mathrm{siconc}'
        r'$'
    ),
    (
        '5.4',
        'flux_density_y',
        r'$F_y = v_{\mathrm{si}} \times \mathrm{sithick} \times \mathrm{siconc}'
        r'$'
    ),

    # ── Section 5.3.1: Face flux interpolation ───────────────────────────────
    # The flux at each cell face is estimated as the average of the two
    # neighbouring cell-centre flux densities.
    (
        '5.5',
        'face_flux_east_west',
        r'$\bar{F}_{\mathrm{E}}(i) = \frac{1}{2}\left[F_x(i) + F_x(i+1)\right],'
        r'\quad \bar{F}_{\mathrm{W}}(i) = \frac{1}{2}\left[F_x(i) + F_x(i-1)\right]$'
    ),
    (
        '5.6',
        'face_flux_north_south',
        r'$\bar{F}_{\mathrm{N}}(j) = \frac{1}{2}\left[F_y(j) + F_y(j+1)\right],'
        r'\quad \bar{F}_{\mathrm{S}}(j) = \frac{1}{2}\left[F_y(j) + F_y(j-1)\right]$'
    ),

    # ── Section 5.3.2: Full divergence in flux form ───────────────────────────
    # Net volume flux into the cell = (face flux difference) x (face width).
    # Units: m2 s-1 x m = m3 s-1.
    (
        '5.7',
        'divergence_flux_form',
        r'$\mathrm{div}(i,j) = '
        r'\left(\bar{F}_{\mathrm{E}} - \bar{F}_{\mathrm{W}}\right)\Delta y'
        r'\ +\ \left(\bar{F}_{\mathrm{N}} - \bar{F}_{\mathrm{S}}\right)\Delta x(\varphi)'
        r'$'
    ),

    # ── Section 5.3.3: Face width ─────────────────────────────────────────────
    (
        '5.8',
        'face_width',
        r'$\Delta x(\varphi) = '
        r'\mathrm{haversine}\!\left(\varphi,\,0^\circ,\,\varphi,\,\Delta\lambda^\circ\right)'
        r'$'
    ),

    # ── Section 5.3.4: Sign Convention ───────────────────────────────────────
    (
        '5.9',
        'dynamic_sign',
        r'$F_{\mathrm{dynamic,\,cell}} = -\,\mathrm{div}$'
    ),

    # ── Section 5.4: Thermodynamic Residual ──────────────────────────────────
    (
        '5.10',
        'thermo_residual',
        r'$F_{\mathrm{thermo,\,cell}} '
        r'= \Delta \mathrm{SIV}_{\mathrm{cell}} - F_{\mathrm{dynamic,\,cell}}$'
    ),

    # ── Section 6.1: Domain-Integrated Budget ────────────────────────────────
    (
        '6.1',
        'domain_budget',
        r'$\Delta \mathrm{SIV}_{\mathrm{domain}} '
        r'= \sum \Delta \mathrm{SIV}_{\mathrm{cell}} '
        r'= \sum F_{\mathrm{dynamic,\,cell}} + \sum F_{\mathrm{thermo,\,cell}}$'
    ),

    # ── Section 6.2: Sign-Partitioned Growth and Melt ────────────────────────
    (
        '6.2',
        'growth_cell',
        r'$\mathrm{Growth}_{\mathrm{cell}}(t) '
        r'= \max\!\left(F_{\mathrm{thermo,\,cell}}(t),\;0\right)$'
    ),
    (
        '6.3',
        'melt_cell',
        r'$\mathrm{Melt}_{\mathrm{cell}}(t) '
        r'= \min\!\left(F_{\mathrm{thermo,\,cell}}(t),\;0\right)$'
    ),
    (
        '6.4',
        'growth_domain',
        r'$\mathrm{Growth}_{\mathrm{domain}}(t) '
        r'= \sum_{i,j} \max\!\left(F_{\mathrm{thermo},\,i,j}(t),\;0\right)'
        r'$'
    ),
    (
        '6.5',
        'melt_domain',
        r'$\mathrm{Melt}_{\mathrm{domain}}(t) '
        r'= \sum_{i,j} \min\!\left(F_{\mathrm{thermo},\,i,j}(t),\;0\right)'
        r'$'
    ),
    (
        '6.6',
        'import_domain',
        r'$\mathrm{Import}_{\mathrm{domain}}(t) '
        r'= \sum_{i,j} \max\!\left(F_{\mathrm{dynamic},\,i,j}(t),\;0\right)$'
    ),
    (
        '6.7',
        'export_domain',
        r'$\mathrm{Export}_{\mathrm{domain}}(t) '
        r'= \sum_{i,j} \min\!\left(F_{\mathrm{dynamic},\,i,j}(t),\;0\right)$'
    ),

    # ── Additional equations ──────────────────────────────────────────────────

    # Freshwater transport — GLORYS12 discrete form
    # Integrates velocity × FW fraction over depth and zonal width simultaneously,
    # rather than computing FWC first and multiplying by velocity separately.
    (
        'A.0',
        'freshwater_transport_glorys',
        r'$\mathrm{FWT} = \int\!\!\int v_n \,\frac{S_{\mathrm{ref}} - S}{S_{\mathrm{ref}}}\; dz\; dx$'
    ),

    # Freshwater content (integrated over depth)
    (
        'A.0a',
        'freshwater_content',
        r'$\mathrm{FWC} = \int \frac{S_{\mathrm{ref}} - S(z)}{S_{\mathrm{ref}}}\; dz$'
    ),

    # Freshwater volume per cell
    (
        'A.0b',
        'freshwater_volume_cell',
        r'$V_{\mathrm{fw}} = \mathrm{FWC} \times A_{\mathrm{cell}}$'
    ),

    # Freshwater mass transport (integrated over depth and cross-section width)
    (
        'A.1',
        'freshwater_transport',
        r'$F_{\mathrm{WM}} = \int\!\!\int v_n \cdot \mathbf{1}_{\mathrm{WM}}\; dz\; dx$'
    ),

    # Surface energy balance: net heat flux decomposition
    (
        'A.2',
        'net_heat_flux',
        r'$Q_{\mathrm{net}} = Q_{\mathrm{SW}} + Q_{\mathrm{LW}} + Q_{\mathrm{SH}} + Q_{\mathrm{LH}}$'
    ),

    # Cold-air outbreak index
    (
        'A.3',
        'cao_index',
        r'$\mathrm{CAO} = \theta_{\mathrm{skin}} - \theta_{850}$'
    ),

    # Potential temperature
    (
        'A.4',
        'potential_temperature',
        r'$\theta = T \left(\frac{p_0}{p}\right)^{R/c_p}$'
    ),
]


# ── Rendering function ────────────────────────────────────────────────────────

def render_equation(number, label, latex, filename, width=FIG_WIDTH,
                    height=FIG_HEIGHT, fontsize=FONTSIZE, dpi=DPI):
    """Render a single LaTeX equation to a PNG file."""
    fig, ax = plt.subplots(figsize=(width, height))
    fig.patch.set_facecolor(BG_COLOR)
    ax.set_axis_off()
    ax.text(
        0.5, 0.5, latex,
        transform=ax.transAxes,
        fontsize=fontsize,
        ha='center', va='center',
        color=TEXT_COLOR,
    )
    fig.savefig(filename, dpi=dpi, bbox_inches='tight',
                facecolor=BG_COLOR, edgecolor='none',
                pad_inches=0.08)
    plt.close(fig)


# ── Render all individual equations ──────────────────────────────────────────

print(f"Rendering {len(EQUATIONS)} equations to {OUTPUT_DIR}/")
print()

for number, label, latex in EQUATIONS:
    fname = OUTPUT_DIR / f'eq_{number.replace(".", "_")}_{label}.png'
    render_equation(number, label, latex, fname)
    print(f"  ({number}) {label:30s}  ->  {fname.name}")

print()

# ── Reference sheet ───────────────────────────────────────────────────────────
# Single figure showing all equations numbered, for use as a lookup sheet.
# Uses data coordinates (not axes-fraction) throughout so axhline works
# without a transform argument.

N          = len(EQUATIONS)
ref_width  = 9.5
row_height = 0.78
ref_height = N * row_height + 0.8   # extra for title

fig_ref, ax_ref = plt.subplots(figsize=(ref_width, ref_height))
fig_ref.patch.set_facecolor('white')
ax_ref.set_facecolor('white')
ax_ref.set_xlim(0, ref_width)
ax_ref.set_ylim(0, ref_height)
ax_ref.set_axis_off()

# Title — placed in data coordinates
ax_ref.text(ref_width / 2, ref_height - 0.25,
            'SIV Budget Equations \u2014 Reference Sheet (Sections 4\u20136)',
            ha='center', va='top', fontsize=13, fontweight='bold',
            color='#1B3A5C')

# Dividing line under title — y in data coordinates, no transform needed
ax_ref.axhline(ref_height - 0.52, color='#AAAAAA', lw=1.0)

# Each equation
for idx, (number, label, latex) in enumerate(EQUATIONS):
    y = ref_height - 0.75 - (idx + 0.5) * row_height

    # Equation number (left margin)
    ax_ref.text(0.15, y, f'({number})',
                ha='left', va='center', fontsize=9.5,
                color='#666666', fontfamily='monospace')

    # Equation (centred)
    ax_ref.text(ref_width / 2, y, latex,
                ha='center', va='center', fontsize=11.5,
                color='black')

    # Light separator between rows
    if idx < N - 1:
        ax_ref.axhline(ref_height - 0.75 - (idx + 1) * row_height,
                       color='#EEEEEE', lw=0.7)

ref_path = OUTPUT_DIR / 'eq_reference_sheet.png'
fig_ref.savefig(ref_path, dpi=DPI, bbox_inches='tight',
                facecolor='white', edgecolor='none', pad_inches=0.12)
plt.close(fig_ref)
print(f"Reference sheet  ->  {ref_path.name}")
print()
print("Done. Insert individual eq_*.png files into Word at 12 cm width.")
print("Use eq_reference_sheet.png as a lookup guide.")

Rendering 25 equations to outputs/equations/

  (4.1) budget_equation                 ->  eq_4_1_budget_equation.png
  (5.1) siv_per_cell                    ->  eq_5_1_siv_per_cell.png
  (5.2) dsiv_per_cell                   ->  eq_5_2_dsiv_per_cell.png
  (5.3) flux_density_x                  ->  eq_5_3_flux_density_x.png
  (5.4) flux_density_y                  ->  eq_5_4_flux_density_y.png
  (5.5) face_flux_east_west             ->  eq_5_5_face_flux_east_west.png
  (5.6) face_flux_north_south           ->  eq_5_6_face_flux_north_south.png
  (5.7) divergence_flux_form            ->  eq_5_7_divergence_flux_form.png
  (5.8) face_width                      ->  eq_5_8_face_width.png
  (5.9) dynamic_sign                    ->  eq_5_9_dynamic_sign.png
  (5.10) thermo_residual                 ->  eq_5_10_thermo_residual.png
  (6.1) domain_budget                   ->  eq_6_1_domain_budget.png
  (6.2) growth_cell                     ->  eq_6_2_growth_cell.png
  (6.3) melt_cell                  